In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
import cv2
import joblib

from skimage.color import rgb2gray
from skimage.feature import hog

from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score

Device and Classes


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]


Load test dataset

In [ ]:
transform = transforms.ToTensor()

test_dataset = torchvision.datasets.CIFAR10(
    root="../data",
    train=False,
    download=False,
    transform=transform
)

print("Test images:", len(test_dataset))

Image degradation functions

Gaussian noise

In [ ]:
def add_gaussian_noise(image, std=0.1):
    noise = torch.randn_like(image) * std

    noisy_image = image + noise

    noisy_image = torch.clamp(
        noisy_image,
        0.0,
        1.0
    )

    return noisy_image

Gaussian blur

In [ ]:
def add_blur(image, kernel_size=5):

    image_np = image.permute(1, 2, 0).numpy()

    blurred = cv2.GaussianBlur(
        image_np,
        (kernel_size, kernel_size),
        0
    )

    blurred = torch.tensor(
        blurred,
        dtype=torch.float32
    ).permute(2, 0, 1)

    return blurred

Rotation


In [ ]:
def rotate_image(image, angle=20):

    image_np = image.permute(1, 2, 0).numpy()

    height, width = image_np.shape[:2]

    center = (width // 2, height // 2)

    rotation_matrix = cv2.getRotationMatrix2D(
        center,
        angle,
        1.0
    )

    rotated = cv2.warpAffine(
        image_np,
        rotation_matrix,
        (width, height)
    )

    rotated = torch.tensor(
        rotated,
        dtype=torch.float32
    ).permute(2, 0, 1)

    return rotated

Occlusion


In [ ]:
def add_occlusion(image, size=10):

    occluded = image.clone()

    _, height, width = occluded.shape

    start_x = width // 2 - size // 2
    start_y = height // 2 - size // 2

    occluded[
        :,
        start_y:start_y + size,
        start_x:start_x + size
    ] = 0

    return occluded

Visualize degradations

In [ ]:
image, label = test_dataset[0]

noisy = add_gaussian_noise(image, std=0.15)
blurred = add_blur(image, kernel_size=5)
rotated = rotate_image(image, angle=20)
occluded = add_occlusion(image, size=10)

images = [
    image,
    noisy,
    blurred,
    rotated,
    occluded
]

titles = [
    "Original",
    "Noise",
    "Blur",
    "Rotation",
    "Occlusion"
]

fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for ax, img, title in zip(axes, images, titles):

    ax.imshow(img.permute(1, 2, 0))

    ax.set_title(title)

    ax.axis("off")

plt.tight_layout()
plt.show()

Load CNN

Define same CNN architecture

In [ ]:
class SimpleCNN(nn.Module):

    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256
            ),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(
                256,
                10
            )
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x

Load CNN weights

In [ ]:
cnn_model = SimpleCNN().to(device)

cnn_model.load_state_dict(
    torch.load(
        "../results/simple_cnn_cifar10.pth",
        map_location=device
    )
)

cnn_model.eval()

print("CNN model loaded.")

Load classical model

In [ ]:
svm_model = joblib.load(
    "../results/hog_svm_cifar10.pkl"
)

print("HOG + SVM model loaded.")

CNN robustness evaluation

CNN evaluation function

In [ ]:
def evaluate_cnn(dataset, degradation=None, max_samples=2000):

    correct = 0
    total = 0

    cnn_model.eval()

    with torch.no_grad():

        for i in range(min(max_samples, len(dataset))):

            image, label = dataset[i]

            if degradation is not None:
                image = degradation(image)

            image = image.unsqueeze(0).to(device)

            output = cnn_model(image)

            prediction = output.argmax(dim=1).item()

            if prediction == label:
                correct += 1

            total += 1

    return correct / total

HOG + SVM robustness evaluation

In [ ]:
def evaluate_svm(dataset, degradation=None, max_samples=2000):

    predictions = []
    labels = []

    for i in range(min(max_samples, len(dataset))):

        image, label = dataset[i]

        if degradation is not None:
            image = degradation(image)

        image_np = image.permute(1, 2, 0).numpy()

        gray = rgb2gray(image_np)

        features = hog(
            gray,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            feature_vector=True
        )

        prediction = svm_model.predict(
            [features]
        )[0]

        predictions.append(prediction)
        labels.append(label)

    return accuracy_score(
        labels,
        predictions
    )

Run experiments

Clean images

In [ ]:
cnn_clean = evaluate_cnn(
    test_dataset
)

svm_clean = evaluate_svm(
    test_dataset
)

print("CNN Clean Accuracy:", cnn_clean)
print("HOG + SVM Clean Accuracy:", svm_clean)

Noise

In [ ]:
noise_function = lambda x: add_gaussian_noise(
    x,
    std=0.15
)

cnn_noise = evaluate_cnn(
    test_dataset,
    noise_function
)

svm_noise = evaluate_svm(
    test_dataset,
    noise_function
)

print("CNN Noise Accuracy:", cnn_noise)
print("HOG + SVM Noise Accuracy:", svm_noise)

BLUR

In [ ]:
blur_function = lambda x: add_blur(
    x,
    kernel_size=5
)

cnn_blur = evaluate_cnn(
    test_dataset,
    blur_function
)

svm_blur = evaluate_svm(
    test_dataset,
    blur_function
)

print("CNN Blur Accuracy:", cnn_blur)
print("HOG + SVM Blur Accuracy:", svm_blur)

Rotation

In [ ]:
rotation_function = lambda x: rotate_image(
    x,
    angle=20
)

cnn_rotation = evaluate_cnn(
    test_dataset,
    rotation_function
)

svm_rotation = evaluate_svm(
    test_dataset,
    rotation_function
)

print("CNN Rotation Accuracy:", cnn_rotation)
print("HOG + SVM Rotation Accuracy:", svm_rotation)

Occlusion

In [ ]:
occlusion_function = lambda x: add_occlusion(
    x,
    size=10
)

cnn_occlusion = evaluate_cnn(
    test_dataset,
    occlusion_function
)

svm_occlusion = evaluate_svm(
    test_dataset,
    occlusion_function
)

print("CNN Occlusion Accuracy:", cnn_occlusion)
print("HOG + SVM Occlusion Accuracy:", svm_occlusion)

Collect results

In [ ]:
conditions = [
    "Clean",
    "Noise",
    "Blur",
    "Rotation",
    "Occlusion"
]

cnn_results = [
    cnn_clean,
    cnn_noise,
    cnn_blur,
    cnn_rotation,
    cnn_occlusion
]

svm_results = [
    svm_clean,
    svm_noise,
    svm_blur,
    svm_rotation,
    svm_occlusion
]

for condition, cnn_acc, svm_acc in zip(
    conditions,
    cnn_results,
    svm_results
):

    print(
        f"{condition:10s} | "
        f"CNN: {cnn_acc:.4f} | "
        f"HOG+SVM: {svm_acc:.4f}"
    )

Visual comparison

In [ ]:
x = np.arange(len(conditions))

width = 0.35

plt.figure(figsize=(10, 5))

plt.bar(
    x - width / 2,
    svm_results,
    width,
    label="HOG + SVM"
)

plt.bar(
    x + width / 2,
    cnn_results,
    width,
    label="CNN"
)

plt.xticks(
    x,
    conditions
)

plt.ylabel("Accuracy")

plt.title(
    "Classical vs Deep Vision Under Image Degradation"
)

plt.legend()

plt.tight_layout()

plt.show()